In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
import os
import time

In [9]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.8
DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
MODEL_NAME = "Logistic Regression (SMOTE)"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"

In [3]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [4]:
# Split data into features and target
X = df.drop('Output', axis=1)
y = df['Output']

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

# Split into train and test before SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("\nApplying SMOTE to the training data...")
count_of_majority_class = y_train.value_counts()[0]
print(f"Number of samples in the majority class (0) in y_train: {count_of_majority_class}")
desired_strategy = {
    1: count_of_majority_class,
    2: count_of_majority_class,
    3: count_of_majority_class
}

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=desired_strategy)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Resampled training features shape: {X_train_resampled.shape}")
print(f"Resampled training target shape: {y_train_resampled.shape}")
print(f"Class distribution in resampled y_train:\n{y_train_resampled.value_counts(normalize=True)}")


Features (X) shape: (510530, 13)
Target (y) shape: (510530,)

Applying SMOTE to the training data...
Number of samples in the majority class (0) in y_train: 79565
Resampled training features shape: (318260, 13)
Resampled training target shape: (318260,)
Class distribution in resampled y_train:
Output
0    0.25
3    0.25
1    0.25
2    0.25
Name: proportion, dtype: float64


In [5]:
# Train the Logistic Regression model

model = LogisticRegression(
    solver='saga',
    max_iter=2000,
    random_state=RANDOM_STATE,
)

train_start_time = time.time()
model.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - train_start_time

C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [6]:
# Evaluate the model

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 32.26%
F1 Score: 0.3888

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.30      0.45    318260
           1       0.10      0.26      0.14     29166
           2       0.12      0.28      0.17     35386
           3       0.12      0.69      0.21     25612

    accuracy                           0.32    408424
   macro avg       0.30      0.38      0.24    408424
weighted avg       0.70      0.32      0.39    408424



In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference


_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
_timing = time_inference(model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="SMOTE",
    labels=[0, 1, 2, 3],
    class_names=["benign", "simple", "intermediate", "sophisticated"],
    y_proba=_proba,
    y_train=y_train_resampled,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=model.get_params(),
    seed=RANDOM_STATE,
    n_features=X_test.shape[1],
    n_train=len(y_train_resampled),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")